# Linear probes: 잘 맞는 분류기에서 믿을 수 있는 실험으로

**대상:** ARENA 1.2를 마쳤고 residual stream·hook·activation patching을 아는 학습자.
**목표:** probe를 구현하는 것보다, 결과가 무엇을 입증하고 무엇을 입증하지 않는지 판단하기.
원본 `[1.3.1] Linear Probes`를 한국어 개인 학습용으로 재구성했다. 원본은 보존했다.

시작 질문: **held-out accuracy 95%인 truth probe를 발견했다. 이 모델을 거짓말 탐지기로 써도 되는가?**
지금 답을 두 문장으로 써 두고 마지막에 고쳐 보자.

In [1]:
first_answer = ""  # 관측 사실 / 추가로 필요한 증거를 각각 한 문장으로

## 진행 순서 — 약 4~6시간, 중간에 멈춰도 됨

| 순서 | 파일 | 핵심 실험 질문 | 예상 시간 |
|---|---|---|---|
| 1 | `01_design_and_probes.ipynb` | 높은 점수가 label 대신 shortcut을 읽은 결과라면? | 60~90분 |
| 2 | `02_real_truth_experiment.ipynb` | 사실·주제가 바뀌어도 같은 방향이 통하는가? | 60~90분 |
| 3 | `03_intervention_and_claims.ipynb` | 읽을 수 있다는 것과 출력에 영향을 준다는 것은 같은가? | 45~60분 |
| 4 | `04_deception_attention.ipynb` | truth, 지시문, 실제 deception, high-stakes를 어떻게 구분하는가? | 60~90분 |

각 노트북은 **예측 → 실험 → 반례 → 허용되는 결론** 순서다. 출력 전에 예측을 적어야 실험이 학습이 된다.
막히면 해당 질문의 접힌 힌트를 먼저 보고, `probe_lab.py`는 구현 참고용으로 연다.
기본 학습은 저장된 실제 activation과 작은 합성 실험으로 진행한다. 13B 재실행은 선택이다.

## 먼저 합의할 언어

- **사실의 참/거짓:** 외부 정답에 비추어 라벨을 정한다. 모델의 믿음 자체는 아니다.
- **선형적으로 읽힘:** 지정한 데이터·위치·층에서 선형 분류기가 미지 표본을 구분한다.
- **일반화:** 새 entity, 주제, template, 모델 중 무엇이 바뀌었는지를 명시해야 한다.
- **개입 민감성:** 해당 activation 조작에 출력이 달라진다. 자연 실행의 유일한 인과 메커니즘이라는 뜻은 아니다.
- **기만:** false utterance와 동의어가 아니다. 진실만 골라 말해도 오도할 수 있다.

연구의 최신성은 `SOURCES_AND_CLAIMS.md`에 정리했다. **원본 코드 버그**, **논문의 적용 범위**, **후속 반례**는 서로 구분한다.
한 후속 논문의 성공/실패를 학계 전체의 합의나 완전한 debunk로 부르지 않는다.

In [2]:
from pathlib import Path
import sys
HERE = Path.cwd()
if not (HERE / "probe_lab.py").exists():
    candidates = [HERE / "study/linear_probes_ko", HERE / "ARENA_3.0/study/linear_probes_ko"]
    HERE = next(p for p in candidates if (p / "probe_lab.py").exists())
sys.path.insert(0, str(HERE))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from probe_lab import (group_split_indices, fit_mean_difference, fit_logistic_probe,
                       score_metrics, fit_train_pca, group_bootstrap_ci)
np.set_printoptions(precision=3, suppress=True)

In [3]:
import torch, json, platform
print("Python:", platform.python_version(), "Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU: 캐시 분석은 가능")
print("실제 실험 결과:", (HERE / "artifacts/results.json").exists())
print("실습 폴더:", HERE)

Python: 3.12.14 Torch: 2.11.0+cu128
GPU: NVIDIA RTX A6000
실제 실험 결과: True
실습 폴더: /root/ARENA_3.0/study/linear_probes_ko


## Vast에서 열기 / 중단 / 복귀

VS Code의 커널 선택에서 **ARENA · Vast PyTorch** 또는 `/venv/main/bin/python`을 고른다.
파일을 저장하면 VS Code 종료 후에도 서버 디스크에는 남는다. 서버의 recycle/destroy는 디스크를 지운다.
GPU 모델은 노트북마다 동시에 여러 개 올리지 않는다. 캐시 분석에는 모델이 필요 없다.

현재 챕터는 OpenAI API를 호출하지 않는다. Llama 추론은 이 Vast GPU에서 한다.
HF 토큰은 코드/노트북/`.env`에 넣지 않았다. 이미 다운로드한 모델은 `local_files_only=True`로 읽는다.
다른 인스턴스에서 다시 다운로드할 때만 HF 로그인이 필요하다.

**중간 점검:** 다음 실험을 하기 전 “split 단위 / 선택에 쓰는 데이터 / 단 하나의 주장 / 그 주장을 깨는 결과”를 기록한다.